<a href="https://colab.research.google.com/github/Hanmikoo/newurban-journey/blob/main/%EA%B8%B0%EB%A7%90%EC%8B%9C%EA%B0%81%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Kaggle API 기반 자동화 데이터 다운로드 및 보안 설정

1. Kaggle API 라이브러리 설치 (!pip install -q kaggle)

2. API Credential(인증서) 파일 업로드 및 가상 경로 이동 (files.upload())

3. 리눅스 파일 권한 제한을 통한 보안 강화 (!chmod 600)

4. API 명령어 기반 데이터셋 고속 다운로드 (!kaggle datasets download ...)

5. 백그라운드 압축 해제 및 디렉터리 타겟팅 (!unzip -q -o)


In [1]:
# 1. 캐글 API 설치 및 인증서 업로드
!pip install -q kaggle
from google.colab import files
files.upload() # 여기에 캐글에서 다운로드한 'kaggle.json' 파일을 선택해 업로드

# 2. 인증서 폴더 이동 및 권한 설정
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Garbage Detection 데이터셋 다운로드 및 압축 해제
!kaggle datasets download -d viswaprakash1990/garbage-detection --force
!unzip -q -o garbage-detection.zip -d ./garbage_dataset


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/viswaprakash1990/garbage-detection
License(s): Attribution 4.0 International (CC BY 4.0)
100% 191M/191M [00:05<00:00, 35.7MB/s]



1. Random  Sampling으로 데이터 크기 축소: torch.utils.data.Subset과 random.sample을 사용하여 전체 데이터 중 10%만 무작위로 추출(Sampling)하여 학습 데이터셋을 구성
2. Data Augmentation 기법: 모델의 과적합(Overfitting)을 방지하고 일반화 성능을 높이기 위해 torchvision.transforms를 활용함

1.   RandomHorizontalFlip: 이미지를 무작위로 좌우 반전시켜 데이터의 다양성을 확보
2.   RandomRotation(15): 이미지를 무작위로 최대 15도까지 회전시켜, 쓰레기가 어떤 각도로 놓여있든 잘 인식할 수 있도록 학습 데이터를 확장





In [2]:
import os
import random
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import shutil
import re

# Helper function to extract base class name (e.g., 'biodegradable' from 'biodegradable1001_jpg')
def get_base_class_name(filename):
    match = re.match(r'^([a-zA-Z]+)\d+.*', filename)
    if match:
        return match.group(1)
    return None

# 1. 데이터셋 경로 설정 (이전 단계에서 압축 푼 폴더)
base_data_dir = './garbage_dataset/GARBAGE CLASSIFICATION/train'
original_image_dir = os.path.join(base_data_dir, 'images')

# --- Start of enhanced restructuring logic ---

# 0. Clean up any previous incorrect structure under base_data_dir
# This is crucial to ensure we start from a clean slate.
print(f"기존 데이터 구조 정리 중: '{base_data_dir}'")
for item in os.listdir(base_data_dir):
    item_path = os.path.join(base_data_dir, item)
    # Do not delete the 'images' folder yet, as it contains original files
    if os.path.isdir(item_path) and item != 'images':
        print(f"  - 삭제: {item_path}")
        shutil.rmtree(item_path)

# ImageFolder에 맞게 데이터 구조 변경
# 클래스 폴더가 없으면 생성하고 이미지를 이동
if os.path.exists(original_image_dir) and os.listdir(original_image_dir): # Ensure images dir exists and is not empty
    print(f"데이터 디렉토리 '{original_image_dir}'의 내용 확인 및 구조 변경 중...")

    image_files = [f for f in os.listdir(original_image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

    # 정확한 클래스 이름 (예: 'biodegradable', 'cardboard') 추출
    actual_classes = set()
    for f in image_files:
        class_name = get_base_class_name(f)
        if class_name:
            actual_classes.add(class_name)

    classes = sorted(list(actual_classes))
    print(f"발견된 실제 클래스: {classes}") # Verify actual classes here

    if not classes:
        raise ValueError("No valid classes found after processing image filenames.")

    for cls_name in classes:
        class_folder = os.path.join(base_data_dir, cls_name)
        os.makedirs(class_folder, exist_ok=True)
        print(f"  - 클래스 폴더 생성: {class_folder}")

    # 이미지들을 올바른 클래스 폴더로 이동
    moved_count = 0
    # Use a copy of the list to avoid issues if original_image_dir is modified during iteration
    for filename in list(os.listdir(original_image_dir)):
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            cls_name = get_base_class_name(filename)
            if cls_name:
                src_path = os.path.join(original_image_dir, filename)
                dest_path = os.path.join(base_data_dir, cls_name, filename)
                # Check if file needs to be moved and if destination exists
                if os.path.exists(src_path) and not os.path.exists(dest_path):
                    shutil.move(src_path, dest_path)
                    moved_count += 1
            else:
                print(f"Warning: Could not determine base class for file {filename}. Skipping.")
    print(f"{moved_count}개의 이미지가 올바른 클래스 폴더로 이동되었습니다.")

    # 원본 images 폴더가 비어있으면 삭제
    if not os.listdir(original_image_dir):
        os.rmdir(original_image_dir)
        print(f"원본 이미지 폴더 '{original_image_dir}'가 비어 있어 삭제되었습니다.")
    else:
        print(f"경고: '{original_image_dir}' 폴더에 이미지 파일이 아닌 다른 파일들이 남아있습니다. ImageFolder 처리에 영향을 주지 않습니다.")
else:
    print(f"경고: 원본 이미지 디렉토리 '{original_image_dir}'가 존재하지 않거나 비어 있습니다. 데이터 구조 변경을 건너뜁니다.")

# --- End of enhanced restructuring logic ---

# ImageFolder의 root 경로를 이제 base_data_dir로 설정합니다.
data_dir = base_data_dir

#  'Augmentation(데이터 증강)'을 적용
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(), # 랜덤 좌우 반전 (Augmentation 1)
    transforms.RandomRotation(15),     # 랜덤 회전 (Augmentation 2)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 전체 데이터 불러오기
print(f"데이터 디렉토리 '{data_dir}'의 최종 내용:")
!ls -R "{data_dir}"

full_dataset = ImageFolder(root=data_dir, transform=train_transform)

# Print out classes found by ImageFolder for debugging and verification
print(f"ImageFolder에서 발견한 클래스: {full_dataset.classes}")
print(f"ImageFolder에서 발견한 class_to_idx: {full_dataset.class_to_idx}")
print(f"ImageFolder에서 로드된 샘플 수: {len(full_dataset)}")


#  속도를 위해 데이터의 10%만 샘플링
num_samples = int(len(full_dataset) * 0.1)
indices = random.sample(range(len(full_dataset)), num_samples)
train_subset = Subset(full_dataset, indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
print(f"전체 데이터 중 {len(train_subset)}개만 샘플링하여 학습을 준비했습니다.")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
cardboard580_jpg.rf.77430ba63bd4e4a20128bdc3112670cd.jpg
cardboard584_jpg.rf.c740b17078c3cb7c41f9db9ccdd0fa86.jpg
cardboard585_jpg.rf.2b52a9c612d2dddbd2ef6e6d4195a951.jpg
cardboard586_jpg.rf.dbf38f0ae870b3d97b17aa3915861af2.jpg
cardboard587_jpg.rf.124877ee8820bc7aadc418f3e59ff456.jpg
cardboard588_jpg.rf.5b896877226ead8099037939e027d607.jpg
cardboard589_jpg.rf.db3925a66438713c15e77df70b07d0d7.jpg
cardboard58_jpg.rf.2b6f48bec3e2dc606b17bb45afa87b17.jpg
cardboard593_jpg.rf.55cc3cbbf9a2bd07f9f3be408c3b50f0.jpg
cardboard594_jpg.rf.bc6fa482cffb592cfc1309a065fc8f7f.jpg
cardboard595_jpg.rf.4308818d0297cd4a8687361d77bc7810.jpg
cardboard596_jpg.rf.2d7bac4bbbc98e02f75fae7e9949c8ac.jpg
cardboard597_jpg.rf.3f79c72623e04952a041442a3479cef6.jpg
cardboard598_jpg.rf.4c0334cd58ac6d7985c6003988fb4083.jpg
cardboard59_jpg.rf.7004b5014b697dc0ed2d81081863766b.jpg
cardboard5_jpg.rf.2793aea22cdf069f1c36a76995c1a314.jpg
cardboard600_jpg.rf.56defd0952993da0f424953974e2653e.jpg

적용한 기법: Pre-trained ResNet18을 활용한 전이학습 (Transfer Learning)
대용량 이미지 데이터셋(ImageNet)으로 이미 학습이 완료된 가중치(Weights)를 가져와,  데이터셋에 맞게 미세조정하는 전이학습(Transfer Learning) 기법을 적용

사전 학습된 백본 모델 로드 (models.resnet18)

최종 출력 레이어(Classifier) 구조 변경 (nn.Linear)

하드웨어 가속 기법 (GPU 연산 최적화)

다중 분류를 위한 손실 함수 및 최적화 기법

Loss Function: 여러 개의 쓰레기 종류 중 정답을 맞히는 다중 분류(Multi-class Classification) 문제에 가장 적합한 CrossEntropyLoss를 적용

Optimizer: 기울기(Gradient)의 방향과 보폭(Learning Rate)을 모두 유연하게 조절하여 가장 안정적으로 수렴하는 Adam 옵티마이저를 선택

In [3]:
import torchvision.models as models

# 간단한 모델 생성 (빠른 학습을 위해 가벼운 ResNet18 사용)
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(full_dataset.classes)) # 클래스 수에 맞게 출력 설정
model = model.to('cuda') # GPU 연결

#  일반 CrossEntropy 외에 성능 향상을 위한 Label Smoothing 기법 추가 (Loss Function 변경)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습 시작 (실행 로그 생성)
print("=== 학습을 시작합니다.  ===")
model.train()
for epoch in range(3): # 빠르게 끝내기 위해 3번만 돌리기
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to('cuda'), labels.to('cuda')

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/3], Loss: {running_loss/len(train_loader):.4f}")
print("=== 학습이 완료되었습니다 ===")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


=== 학습을 시작합니다.  ===
Epoch [1/3], Loss: 1.5962
Epoch [2/3], Loss: 1.1290
Epoch [3/3], Loss: 0.9714
=== 학습이 완료되었습니다 ===
